In [ ]:
#| default_exp network

In [ ]:
#| export

from __future__ import annotations

from collections import defaultdict
from typing import Dict, List

from comm_core.backend import CommunicationBackend
from comm_core.core import Message, Node, TransmissionResult
from comm_core.metrics import MetricsCollector
from comm_core.protocol import Protocol
from comm_core.topology import Topology

In [ ]:
#| export

class Network:

    def __init__(
        self,
        nodes: List[Node],
        topology: Topology,
        protocol: Protocol,
        backend: CommunicationBackend,
        metrics: MetricsCollector | None = None,
    ):
        self.nodes = {
            node.id: node
            for node in nodes
        }

        self.topology = topology
        self.protocol = protocol
        self.backend = backend

        self.metrics = (
            metrics
            if metrics is not None
            else MetricsCollector()
        )

        self._inbox: Dict[str, List[Message]] = (
            defaultdict(list)
        )

        self.time = 0.0

    def send(
        self,
        message: Message,
    ) -> None:

        if message.sender not in self.nodes:
            raise ValueError(
                f"Unknown sender: {message.sender}"
            )

        if message.receiver not in self.nodes:
            raise ValueError(
                f"Unknown receiver: {message.receiver}"
            )

        if not self.topology.is_connected(
            message.sender,
            message.receiver,
        ):
            raise ValueError(
                f"No link from "
                f"{message.sender} to "
                f"{message.receiver}"
            )

        self.protocol.submit(
            message,
            self.time,
        )

    def step(
        self,
        dt: float = 1.0,
    ) -> List[TransmissionResult]:

        self.time += dt

        self.backend.step(self.time)

        transmissions = self.protocol.schedule(
            self.time
        )

        results = []

        for transmission in transmissions:

            link = self.topology.get_link(
                transmission.sender,
                transmission.receiver,
            )

            result = self.backend.transmit(
                transmission,
                link,
            )

            self.metrics.record(
                transmission,
                result,
            )

            if result.success:

                self._inbox[
                    transmission.receiver
                ].append(
                    transmission.message
                )

            results.append(result)

        return results

    def receive(
        self,
        node_id: str,
    ) -> List[Message]:

        messages = self._inbox[node_id]

        self._inbox[node_id] = []

        return messages

    

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()